# Retrosynthesis exercise

Using the USPTO-50k dataset, here this exercise addresses the core principle of template-based retrosynthesis:

Product → Reaction template → Reactants

The first part of this notebook deals with only predicting the first step here, the reaction templates, i.e. here simply the reaction class. This simple classification model would become a policy in a multi-step search algorithm.

To make this a bit less abstract, in the second part of the notebook, reactants are recovered from the predicted reaction classes. Note that this is a simple data-driven approach and does include 
- atom mapping
- chemistry rules (hard‑coded)
- graph rewriting.

Hence, it is a conceptual demonstration, far from the prowess of more complex models. However, early systems would have adopted a very similar approach, so there is some real-world relevance of this exercise.

### Part 1: Predicting reaction classes

1) Setup

In [ ]:
import numpy as np
import pandas as pd

from rdkit import Chem
from rdkit.Chem import DataStructs, rdFingerprintGenerator

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

from tqdm import tqdm


2) Load Dataset

In [ ]:
df = pd.read_csv("uspto-50k.csv")
df = df[["prod_smiles", "rxn_smiles", "rxn_class"]]
df["reactants"] = df["rxn_smiles"].str.split(">").str[0]

print("Number of reactions:", len(df))
df.head()

3) Featurisation
In retrosynthesis, only the product is known at the prediction time, so the product needs to be encoded (here: MorganFP).

In [ ]:
# helper function 
def smiles_to_fp(smiles, n_bits=2048, radius=2):
    fpgen = rdFingerprintGenerator.GetMorganGenerator(radius=radius, fpSize=n_bits)
    
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    
    fp = fpgen.GetFingerprint(mol)
    fp_arr = np.zeros((n_bits,), dtype=np.int8)
    DataStructs.ConvertToNumpyArray(fp, fp_arr)
    return fp_arr

In [ ]:
# Build the dataset
X = []
y = []
row_indices = [] 

for idx, row in df.iterrows():
    fp = smiles_to_fp(row["prod_smiles"])
    if fp is None:
        continue
    X.append(fp)
    y.append(row["rxn_class"])
    row_indices.append(idx)

X = np.array(X)
y = np.array(y)
row_indices = np.array(row_indices)

print("Usable samples:", len(X))
print("Number of classes:", len(set(y)))

In [ ]:
# Optional subsampling if too high load:
MAX_SAMPLES = 10000  # 30k took about 10 mins of training on my PC, 10k about 2.5 mins

if MAX_SAMPLES:
    idx = np.random.choice(len(X), MAX_SAMPLES, replace=False)
    X = X[idx]
    y = y[idx]
    row_indices = row_indices[idx]


X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(
    X, y, row_indices,
    test_size=0.2,
    random_state=42,
    stratify=y
)


print("Train size:", len(X_train))
print("Test size:", len(X_test))

4) Train retrosynthesis classifier
Multinomial logistic regression is fast, interpretable, probabilistic (top-k!). Other models could be used as well, e.g. RF.

In [ ]:
model = LogisticRegression(
    max_iter=1000,
    n_jobs=-1,
    solver="saga"
)

model.fit(X_train, y_train)

5) Evaluate top-k accuracy 

In [ ]:
def top_k_accuracy(y_true, probs, k):
    top_k = np.argsort(probs, axis=1)[:, -k:]
    return np.mean([
        y_true[i] in top_k[i]
        for i in range(len(y_true))
    ])

In [ ]:
probs = model.predict_proba(X_test)

for k in [1, 3, 5]:
    acc = top_k_accuracy(y_test, probs, k)
    print(f"Top-{k} accuracy: {acc:.3f}")

6) Inspect some results:

In [ ]:
i = np.random.randint(len(X_test))

true_class = y_test[i]
predicted_classes = np.argsort(probs[i])[-5:][::-1]

print("True reaction class:", true_class)
print("Top-5 predicted classes:", predicted_classes.tolist())

7) Discussion

- Why is retrosynthesis naturally a *ranking* problem?
- Why is accuracy higher than forward prediction?
- What chemical information is missing?
- How would this integrate into a multi-step planner?

### Part 2: Approximating the reactants

Idea: For each reaction class...
- collect all reactant SMILES
- keep the most frequent ones
- later, retrieve from this list

Result: Fast and robust approximation

1) Build reactant libraries per class

In [ ]:
from collections import defaultdict, Counter

class_to_reactants_freq = defaultdict(list)

for cls, idx in zip(y_train, idx_train):
    reactants = df.loc[idx, "reactants"]
    class_to_reactants_freq[int(cls)].append(reactants)

for cls in class_to_reactants_freq:
    counts = Counter(class_to_reactants_freq[cls])
    class_to_reactants_freq[cls] = counts.most_common()


Inspect reaction classes:

In [ ]:
example_class = list(class_to_reactants_freq.keys())[0]

print("Reaction class:", example_class)
print("Most common reactants:")
for r, count in class_to_reactants_freq[example_class][:5]:
    print(f"- {r} ({count}×)")


2) Predict Reactants Given a Predicted Reaction Class

In [ ]:
"""
    Return top-k most frequent reactant sets
    for a given reaction class.
    """

def predict_reactants(predicted_class, k=5):
    predicted_class = int(predicted_class)
    
    if predicted_class not in class_to_reactants_freq:
        return []  # or raise a warning
    
    return [
        r for r, _ in class_to_reactants_freq[predicted_class][:k]
    ]


3) Combine with the class prediction:

In [ ]:
i = np.random.randint(len(X_test))

# True values
true_class = y_test[i]

# CORRECT: map index -> class label
predicted_class = model.classes_[np.argmax(probs[i])]

predicted_reactants = predict_reactants(predicted_class, k=5)

print("True reaction class:", true_class)
print("Predicted reaction class:", predicted_class)

print("\nTop-5 predicted reactant sets:")
for r in predicted_reactants:
    print("-", r)

4) Evaluation: Were the true reactants recovered? 

Use top-k retrieval accuracy:

In [ ]:
def reactant_top_k_accuracy(
    y_true_classes,
    predicted_classes,
    true_reactants,
    k=5
):
    hits = 0
    for cls, true_r in zip(predicted_classes, true_reactants):
        candidates = predict_reactants(cls, k=k)
        if true_r in candidates:
            hits += 1
    return hits / len(true_reactants)

In [ ]:
# Predict reaction classes for test set
predicted_classes = np.argmax(probs, axis=1)

true_reactants_test = df.loc[idx_test, "reactants"].values

for k in [1, 3, 5]:
    acc = reactant_top_k_accuracy(
        y_test,
        predicted_classes,
        true_reactants_test,
        k=k
    )
    print(f"Reactant Top-{k} accuracy: {acc:.3f}")

5) Discussion

- Why is reactant prediction harder than reaction-class prediction?
- Why does top‑k accuracy matter more than top‑1?
- Why don't we retrieve our true reactants? How could this be remedied?
